# SatQuery AI — Stage 1: BigEarthNet Vision-Language Adaptation (QLoRA)

**Base model:** `Qwen/Qwen2-VL-2B-Instruct` (2B, fits T4 ~15GB VRAM)  
**Method:** QLoRA — 4-bit NF4 base + LoRA adapters (PEFT)  
**Hardware:** Google Colab free-tier T4 (15GB VRAM). Fallback: Kaggle T4×2 (`/kaggle/working` instead of Drive).

> **Constraints from `AGENTS.md` — do not simplify away:**
> - QLoRA only (never full fine-tune), small VLM 2–3B only
> - Checkpoint to Drive every ~25 steps + auto-resume (safe to rerun top-to-bottom after disconnect)
> - Dataset subsets cached to Drive once, never re-download full BigEarthNet per session
> - `batch_size=1` + `gradient_accumulation=8` (fits T4), gradient checkpointing ON
> - Each stage has its own notebook + own checkpoint dir; stage 2/3 load previous adapter

---
**How to use:** `Runtime → Run all` (or run cells top-to-bottom). After any disconnect, just run again — it resumes from the latest Drive checkpoint automatically.  
**Expected time:** ~15–25 min for the default 800-sample subset, 1 epoch, on T4.


## 1 — Verify GPU (T4, CUDA, VRAM)

If this shows **Tesla T4** and `torch.cuda.is_available() == True`, you are good. If no GPU: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
import sys, platform, torch
!nvidia-smi  2>&1 | head -n 20
print(f"Python: {sys.version.split()[0]} | platform: {platform.platform()}")
print(f"Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Compute capability: {props.major}.{props.minor} (T4 is 7.5)")
    print(f"Total VRAM: {props.total_memory/1024**3:.1f} GB")
    # T4 is sm_75, does NOT support bf16 efficiently — we will use fp16
    print(f"bf16 supported: {torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown (assume False on T4)'}")
else:
    print("⚠️ No GPU detected — switch runtime to T4 before continuing.")
    # Do not raise — let user fix runtime


## 2 — Mount Google Drive (checkpoints + dataset cache)

All checkpoints and the cached subset live on Drive so training survives disconnects.

- `DRIVE_ROOT = /content/drive/MyDrive/SatQueryAI` (change if you prefer another path)
- `CHECKPOINT_DIR = DRIVE_ROOT/checkpoints/stage1_bigearthnet_qlora`
- `DATASET_CACHE_DIR = DRIVE_ROOT/datasets/bigearthnet_subset`

On Kaggle, replace `/content/drive/MyDrive/...` with `/kaggle/working/SatQueryAI`.

In [ ]:
from pathlib import Path
import os

# --- CONFIGURE THESE IF NEEDED ---
DRIVE_ROOT = Path("/content/drive/MyDrive/SatQueryAI")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "stage1_bigearthnet_qlora"
DATASET_CACHE_DIR = DRIVE_ROOT / "datasets" / "bigearthnet_subset"
ADAPTER_OUTPUT_DIR = CHECKPOINT_DIR / "final_adapter"

# Mount (no-op if already mounted)
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive mounted.")
except ImportError:
    print("Not in Colab (e.g. local/Kaggle) — skipping drive.mount. Set DRIVE_ROOT accordingly.")

for p in [DRIVE_ROOT, CHECKPOINT_DIR, DATASET_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"{p} -> exists={p.exists()}")

print(f"\nCHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"DATASET_CACHE_DIR: {DATASET_CACHE_DIR}")
# List existing checkpoints (if any)
!ls -lh "{CHECKPOINT_DIR}" 2>&1 | head -n 30


## 3 — Install dependencies (restart-free, T4-safe)

**Goal:** avoid the classic Colab breakage `libnvJitLink.so.13` / `CUDA SETUP ERROR` that happens when `pip install -U` upgrades `torch` from `cu128` → `cu130`.

- We **do not upgrade `torch`** — keep Colab's preinstalled `torch 2.10+cu128` (or 2.8+cu126). It already satisfies all deps.
- We pin `transformers`, `peft`, `accelerate`, `bitsandbytes`, `datasets`, `qwen-vl-utils`, `trl` to mutually compatible wheels tested on T4.
- If you see `ImportError: cannot import name 'sync_gpu'` after install, you forgot to restart — just `Runtime → Restart runtime` and re-run from here.

> **After this cell finishes, do `Runtime → Restart runtime` if the output says "Restart required" or if `import bitsandbytes` fails. Then re-run cells from §1 again (they are idempotent).**

In [ ]:
# Check current torch before touching anything
import torch
print(f"Before install — torch {torch.__version__} | cuda {torch.version.cuda if hasattr(torch.version,'cuda') else 'unknown'}")

# Install pinned, T4-tested versions WITHOUT upgrading torch/nvidia libs.
# These versions are known to work together on Colab Python 3.11, CUDA 12.8, T4 (sm_75).
# If you need newer features, bump them together and keep them in this single pip line.
!pip install -q \
    "transformers==4.46.3" \
    "peft==0.14.0" \
    "accelerate==1.4.0" \
    "bitsandbytes==0.45.5" \
    "datasets==3.1.0" \
    "qwen-vl-utils==0.0.10" \
    "trl==0.15.2" \
    "pillow>=10.0.0" \
    "numpy<2.0"

print("\nInstall done. Checking imports (no restart needed if all import ok)...")
import importlib
for pkg in ["transformers","peft","accelerate","bitsandbytes","datasets","qwen_vl_utils","trl"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {getattr(m,'__version__','ok')}")
    except Exception as e:
        print(f"  {pkg}: FAILED — {e}")

# Quick bitsandbytes CUDA sanity check
!python -m bitsandbytes 2>&1 | head -n 30


### 3b — If you hit CUDA / bitsandbytes errors, run this cell then restart

Common symptoms and fixes:
- `CUDA SETUP ERROR: libnvJitLink.so.13 cannot open` → you upgraded to `torch+cu130` but Colab still has `cu12` libs. Fix: `pip uninstall nvidia-nvjitlink-cu12` or reinstall torch `cu128` (uncomment below).
- `cannot import name 'sync_gpu'` → version mix from upgrading without restart. Fix: restart runtime.
- `libcusparse.so.11` → bitsandbytes compiled for different CUDA. Fix: reinstall pinned `bitsandbytes==0.45.5` and restart.

Normally you can skip this cell. Only run if the previous cell's import check failed.

In [ ]:
# >>> SKIP THIS CELL unless you saw an error in §3 <<<
# Uncomment ONE of the fixes below if needed, then Runtime → Restart runtime

# Fix 1: clean mixed CUDA 12/13 jitlink (most common when torch was upgraded to cu130)
# !pip uninstall -y nvidia-nvjitlink-cu12 nvidia-nvjitlink-cu13 2>&1 | tail -n 5

# Fix 2: force torch back to Colab's default cu128 wheel (if you accidentally got cu130)
# !pip uninstall -y torch torchvision torchaudio 2>&1 | tail -n 5
# !pip install -q "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -n 5

# Fix 3: reinstall bitsandbytes clean
# !pip install -q --force-reinstall "bitsandbytes==0.45.5" 2>&1 | tail -n 5

print("If you uncommented a fix, now do Runtime → Restart runtime, then re-run from §1.")


## 4 — Imports + version audit

Imports are isolated here so a restart clearly fixes any `bitsandbytes` mix. This cell must pass without error before training.

In [ ]:
import os, json, random, math, glob, warnings
from pathlib import Path
from PIL import Image
import numpy as np
import torch

import transformers, peft, accelerate, bitsandbytes, datasets, trl
print(f"transformers {transformers.__version__}")
print(f"peft {peft.__version__}")
print(f"accelerate {accelerate.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"datasets {datasets.__version__}")
print(f"trl {trl.__version__}")
print(f"torch {torch.__version__} | cuda {torch.version.cuda}")

# Must be on GPU for QLoRA
assert torch.cuda.is_available(), "CUDA not available — set Runtime → T4 GPU"
device_name = torch.cuda.get_device_name(0)
print(f"Using GPU: {device_name}")

# T4 is fp16-only for practical purposes (bf16 is slow/emulated)
COMPUTE_DTYPE = torch.float16  # keep fp16 on T4
USE_BF16 = False
print(f"Compute dtype: {COMPUTE_DTYPE} (fp16 on T4, bf16 would be slower)")


## 5 — Training config (single source of truth)

Edit these values to adapt the run. They are also saved to `training/configs/bigearthnet_stage1.json` for reproducibility (mirrors `training/configs/` on disk).

- To resume with a different stage, just change `CHECKPOINT_DIR` in §2 and `ADAPTER_TO_CONTINUE` below.
- `SUBSET_SIZE` controls the cached subset size — start small (e.g. 500–1000) for free Colab, scale up once caching works.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class TrainConfig:
    # Model
    base_model: str = "Qwen/Qwen2-VL-2B-Instruct"
    adapter_to_continue: str = ""  # empty = start from base; else path to previous stage adapter on Drive

    # Data — BigEarthNet adaptation (captioning / land-cover QA)
    subset_size: int = 800          # number of image-caption pairs to cache/use (free-tier safe)
    val_ratio: float = 0.05         # 5% for validation
    image_max_pixels: int = 512*28*28  # ~512² equivalent — balances detail vs VRAM (Qwen2-VL dynamic res)
    image_min_pixels: int = 256*28*28
    max_seq_length: int = 1024      # includes image tokens

    # QLoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

    # Optimization — T4-friendly
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8  # effective batch 8
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05
    num_train_epochs: int = 1
    max_steps: int = -1  # -1 = use epochs; set e.g. 300 for quick smoke test
    weight_decay: float = 0.01
    optim: str = "paged_adamw_8bit"  # 8-bit optimizer saves ~2GB vs adamw
    gradient_checkpointing: bool = True
    max_grad_norm: float = 1.0

    # Checkpointing (frequent saves for Drive)
    save_steps: int = 25
    save_total_limit: int = 2
    logging_steps: int = 5
    eval_steps: int = 50
    seed: int = 3407

CFG = TrainConfig()
print(json.dumps(asdict(CFG), indent=2))

# Persist config to Drive + repo configs/
repo_config_path = Path("/content/drive/MyDrive/SatQueryAI/training_config_stage1.json")
try:
    repo_config_path.parent.mkdir(parents=True, exist_ok=True)
    repo_config_path.write_text(json.dumps(asdict(CFG), indent=2))
    print(f"Saved Drive config: {repo_config_path}")
except Exception as e:
    print(f"Could not save Drive config: {e}")
# Also save to repo-relative path if running locally
try:
    local_cfg = Path("training/configs/bigearthnet_stage1.json")
    local_cfg.parent.mkdir(parents=True, exist_ok=True)
    local_cfg.write_text(json.dumps(asdict(CFG), indent=2))
    print(f"Saved local config: {local_cfg.resolve()}")
except Exception as e:
    print(f"Local save skipped: {e}")


## 6 — Dataset subset & cache (Drive)

BigEarthNet is ~70GB full; we never download it in one Colab session. Instead:

1. **Check `DATASET_CACHE_DIR` on Drive** — if a cached subset exists, load it with `load_from_disk` (instant, no download).
2. Otherwise **build a small subset** (real BigEarthNet via `torchgeo`/`HF` if available, else synthetic fallback for smoke-testing the pipeline) and **save to Drive** for next session.

The subset is stored as a HuggingFace `Dataset` with columns `image` (PIL) and `caption`/`qa` text. Swap the `build_synthetic_dataset` call with your real BigEarthNet loader once you have Drive access — the rest of the notebook is loader-agnostic.

> The synthetic fallback generates random satellite-like images + templated captions so the notebook always runs end-to-end (useful for CI / first-time verification). For real training, replace it with `bigearthnet` parquet/HF loading (see comment below).

In [ ]:
from datasets import Dataset, DatasetDict, load_from_disk
from PIL import Image
import io, random

LAND_COVER_LABELS = ["forest", "urban fabric", "arable land", "pasture", "water bodies", "industrial units", "shrubland", "wetlands"]

def build_synthetic_dataset(n, seed=42):
    """Fallback that always works — random RGB chips + templated RS captions."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        # 224x224 random satellite-like chip (color-biased)
        arr = rng.integers(0, 255, (224,224,3), dtype=np.uint8)
        # bias green for vegetation-like samples
        if i % 3 == 0:
            arr[:,:,1] = np.clip(arr[:,:,1].astype(int) + 30, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
        label = random.choice(LAND_COVER_LABELS)
        caption = f"This Sentinel-2 image shows predominantly {label}. " + random.choice([
            "The land cover is homogeneous.",
            "Patches of vegetation are interspersed.",
            "Urban structures are visible in the corner.",
            "Water bodies reflect in the lower quadrant.",
        ])
        # Two-turn format used for SFT: question -> answer
        question = random.choice([
            "Describe the land cover in this satellite image.",
            "What is the dominant land cover?",
            "Caption this remote sensing image.",
        ])
        rows.append({"image": img, "question": question, "answer": caption, "image_id": f"syn_{i:05d}"})
    return Dataset.from_list(rows)

# Try to load cached subset from Drive
def load_or_build_dataset(cfg, cache_dir: Path):
    cached = cache_dir / "dataset.arrow"  # marker: HF saves as folder
    # HF load_from_disk expects the folder itself
    if (cache_dir / "dataset_info.json").exists() or (cache_dir / "state.json").exists() or any(cache_dir.glob("*.arrow")):
        try:
            print(f"Found cached dataset at {cache_dir} — loading from disk...")
            ds = load_from_disk(str(cache_dir))
            # Handle DatasetDict saved previously
            if isinstance(ds, DatasetDict):
                print(f"Loaded DatasetDict: { {k: len(v) for k,v in ds.items()} }")
                return ds
            print(f"Loaded Dataset: {len(ds)} rows")
            # Retro compatibility: wrap single dataset into train/val split if needed
            if len(ds) >= cfg.subset_size:
                ds = ds.train_test_split(test_size=cfg.val_ratio, seed=cfg.seed)
                return ds
            return DatasetDict({"train": ds})
        except Exception as e:
            print(f"Cache load failed ({e}) — rebuilding...")

    print(f"No cache at {cache_dir} — building subset (n={cfg.subset_size})...")
    # ---- REAL BigEarthNet path (uncomment when you have access) ----
    # Example: HF parquet subset
    # from datasets import load_dataset
    # ds = load_dataset("bigearthnet", split="train", streaming=False)  # requires HF dataset or local parquet
    # ds = ds.shuffle(seed=cfg.seed).select(range(cfg.subset_size))
    # ds = ds.map(lambda ex: {"image": ex["image"], "question": "Describe the land cover.", "answer": ", ".join(ex["labels"])} )
    # -----------------------------------------------------------------
    ds_full = build_synthetic_dataset(cfg.subset_size, seed=cfg.seed)
    ds_split = ds_full.train_test_split(test_size=cfg.val_ratio, seed=cfg.seed)
    # Save to Drive for next session
    try:
        ds_split.save_to_disk(str(cache_dir))
        print(f"Saved subset cache to {cache_dir} (reuse next session, no rebuild)")
    except Exception as e:
        print(f"Could not save cache (Drive full/permission?): {e}")
    return ds_split

datasetDict = load_or_build_dataset(CFG, DATASET_CACHE_DIR)
if isinstance(datasetDict, Dataset):
    datasetDict = DatasetDict({"train": datasetDict})
print(datasetDict)
print("Example:", {k: (v if k!='image' else f"<PIL {v.size}>") for k,v in datasetDict["train"][0].items()})
# Preview image (first row)
display(datasetDict["train"][0]["image"])
print(datasetDict["train"][0]["question"])
print(datasetDict["train"][0]["answer"])


## 7 — Load quantized base model + processor (Qwen2-VL-2B)

`Qwen2VLForConditionalGeneration` with `BitsAndBytesConfig` (NF4, double-quant, fp16 compute). Uses `device_map="auto"` so it shards to GPU.

- If you are continuing from a previous stage (e.g. stage 1 adapter for stage 2 SFT), set `CFG.adapter_to_continue` to the Drive path — the LoRA will be loaded on top of this base in §8.
- Dynamic resolution is capped via `min_pixels`/`max_pixels` to keep VRAM flat on T4.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

print(f"Loading base: {CFG.base_model}")
processor = AutoProcessor.from_pretrained(
    CFG.base_model,
    min_pixels=CFG.image_min_pixels,
    max_pixels=CFG.image_max_pixels,
    trust_remote_code=True,
)
print(f"Processor: min_pixels={CFG.image_min_pixels} max_pixels={CFG.image_max_pixels}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4
    bnb_4bit_use_double_quant=True,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    CFG.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
# Required for k-bit training: cast norms to fp32 + enable grad checkpointing hooks
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
print("Base model loaded in 4-bit + prepared for k-bit training.")
print(f"Model device map: {getattr(model, 'hf_device_map', 'auto')}")
# Show memory
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## 8 — Configure LoRA (QLoRA)

Rank 16 / alpha 32 (scaling 2.0) is the standard for 2B VLMs on T4 — ~14M trainable params (~0.7% of 2B). Target modules cover both attention (`q/k/v/o`) and MLP (`gate/up/down`).

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=list(CFG.lora_target_modules),
    bias="none",
    task_type="CAUSAL_LM",
)
print(lora_config)

# If continuing from previous stage adapter, load it BEFORE wrapping with new LoRA
# Stage 2/3 pattern: base (4-bit) + stage1 adapter (frozen) + new LoRA on top
# For stage 1 this branch is skipped (adapter_to_continue == "")
if CFG.adapter_to_continue and Path(CFG.adapter_to_continue).exists():
    print(f"Continuing from adapter: {CFG.adapter_to_continue}")
    model = PeftModel.from_pretrained(model, CFG.adapter_to_continue, is_trainable=True)
    print("Loaded previous adapter — it will be further fine-tuned.")
else:
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
# Sanity: should be ~0.7% trainable


## 9 — Preprocess dataset for Vision-Language SFT

We format each sample as a Qwen chat with an image token. The processor's `apply_chat_template` produces the correct `image` placeholder + tokenization.

Labels are the assistant answer only (prompt masked with `-100`). `max_seq_length` truncation keeps VRAM flat.

In [ ]:
import copy

def format_sample(example, processor, max_length=CFG.max_seq_length):
    """Return tokenized inputs with labels masked for the user prompt."""
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": example["image"]},
            {"type": "text", "text": example["question"]}
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": example["answer"]}
        ]}
    ]
    # Full chat text (with assistant answer) for labels
    full_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    # Prompt-only text to know where answer starts (for label masking)
    prompt_messages = messages[:1]
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

    full = processor(text=[full_text], images=[example["image"]], padding=False, return_tensors=None)
    prompt = processor(text=[prompt_text], images=[example["image"]], padding=False, return_tensors=None)

    input_ids = full["input_ids"][0]
    # -100 mask for prompt tokens, keep only assistant answer in loss
    labels = copy.deepcopy(input_ids)
    prompt_len = len(prompt["input_ids"][0])
    # Also mask image token placeholders that are part of prompt
    labels[:prompt_len] = [-100] * prompt_len

    # Truncate to max_length (keep suffix which contains the answer)
    if len(input_ids) > max_length:
        input_ids = input_ids[-max_length:]
        labels = labels[-max_length:]

    out = {
        "input_ids": input_ids,
        "attention_mask": [1]*len(input_ids),
        "labels": labels,
    }
    # Qwen2-VL specific: image grid + pixel values are already in full via processor
    if "pixel_values" in full:
        out["pixel_values"] = full["pixel_values"]
    if "image_grid_thw" in full:
        out["image_grid_thw"] = full["image_grid_thw"]
    return out

# Quick test on one sample
sample = datasetDict["train"][0]
tok = format_sample(sample, processor)
print(f"input_ids len: {len(tok['input_ids'])} | labels non-masked: {sum(1 for x in tok['labels'] if x!=-100)}")
print(f"Has pixel_values: {'pixel_values' in tok} | image_grid_thw: {'image_grid_thw' in tok}")
print(processor.decode([x for x in tok["input_ids"] if x!=-100][:120]))

# Map over datasets (batched=False to keep image handling simple). Cache to Drive is NOT needed here — just in-memory map.
def map_fn(example):
    return format_sample(example, processor)

print("Tokenizing train/val (this may take ~1-2 min for 800 samples)...")
train_ds = datasetDict["train"].map(map_fn, remove_columns=datasetDict["train"].column_names, desc="tokenize train")
eval_ds = None
if "test" in datasetDict:
    eval_ds = datasetDict["test"].map(map_fn, remove_columns=datasetDict["test"].column_names, desc="tokenize val")
elif "validation" in datasetDict:
    eval_ds = datasetDict["validation"].map(map_fn, remove_columns=datasetDict["validation"].column_names, desc="tokenize val")
else:
    # datasetDict from train_test_split uses 'test' as val
    eval_ds = datasetDict["test"].map(map_fn, remove_columns=datasetDict["test"].column_names, desc="tokenize val") if "test" in datasetDict else None

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds) if eval_ds else 0}")


## 10 — Training arguments + resume logic

- `save_steps=25` → frequent Drive saves (tolerates disconnects).
- `resume_from_checkpoint` is auto-detected: the latest `checkpoint-*` under `CHECKPOINT_DIR` is reused on every rerun.
- `fp16=True` (not bf16) for T4. `optim=paged_adamw_8bit` saves ~2GB. `gradient_checkpointing=True` trades ~20% speed for ~4GB VRAM.
- `report_to="none"` by default; set to `trackio`/`wandb` if you want experiment tracking.

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import glob, os

# Find latest checkpoint for auto-resume
def find_latest_checkpoint(checkpoint_dir: Path):
    ckpts = sorted(glob.glob(str(checkpoint_dir / "checkpoint-*")), key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1)
    return ckpts[-1] if ckpts else None

latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
print(f"Latest checkpoint: {latest_ckpt if latest_ckpt else '(none — fresh start)'}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_train_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type=CFG.lr_scheduler_type,
    warmup_ratio=CFG.warmup_ratio,
    num_train_epochs=CFG.num_train_epochs,
    max_steps=CFG.max_steps if CFG.max_steps>0 else -1,
    weight_decay=CFG.weight_decay,
    optim=CFG.optim,
    max_grad_norm=CFG.max_grad_norm,
    # Mixed precision
    fp16=True,
    bf16=False,
    # Memory savers
    gradient_checkpointing=CFG.gradient_checkpointing,
    ddp_find_unused_parameters=False,
    # Logging / saving
    logging_steps=CFG.logging_steps,
    save_steps=CFG.save_steps,
    save_total_limit=CFG.save_total_limit,
    save_strategy="steps",
    evaluation_strategy="steps" if eval_ds is not None else "no",
    eval_steps=CFG.eval_steps if eval_ds is not None else None,
    load_best_model_at_end=False,  # keep last, not best — cheaper on Drive
    report_to="none",
    seed=CFG.seed,
    remove_unused_columns=False,  # we have pixel_values / image_grid_thw
    # Hub (optional) — uncomment to push
    # push_to_hub=True, hub_model_id="your-hf-username/satquery-qwen2vl-stage1",
    # For resume: Trainer handles it via resume_from_checkpoint arg below
)
print(training_args)

# Data collator pads input_ids/labels correctly; keeps vision tensors stacked
# For Qwen2-VL the processor already handles image tensors — use default collator with pad
from transformers import Qwen2VLProcessor
data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
# Qwen2-VL needs custom handling for pixel_values: DataCollatorForSeq2Seq ignores them,
# so we wrap it to also pad/stack vision inputs
class VisionDataCollator:
    def __init__(self, base):
        self.base = base
    def __call__(self, features):
        # Separate vision fields before base collator
        has_pixel = "pixel_values" in features[0]
        pixel_values = [f.pop("pixel_values", None) for f in features]
        image_grid_thw = [f.pop("image_grid_thw", None) for f in features]
        batch = self.base(features)
        if has_pixel and pixel_values[0] is not None:
            import torch as _t
            # pixel_values is (num_patches, 3, 14, 14) per sample — stack along batch if same shape, else keep list
            try:
                batch["pixel_values"] = _t.stack([_t.tensor(p) if not isinstance(p,_t.Tensor) else p for p in pixel_values])
            except Exception:
                batch["pixel_values"] = pixel_values
            batch["image_grid_thw"] = _t.tensor(image_grid_thw) if image_grid_thw[0] is not None else None
        return batch

collator = VisionDataCollator(data_collator)
print("TrainingArguments + collator ready.")
print(f"Effective batch size: {CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps}")


## 11 — Trainer

Plain `transformers.Trainer` (not `TRL SFTTrainer`) for maximum version stability on Colab. The resume logic from §10 is passed here.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    tokenizer=processor.tokenizer,  # for save
)
print("Trainer created.")
print(f"Train samples: {len(train_ds)} | Eval: {len(eval_ds) if eval_ds else 0}")
print(f"Max steps: {training_args.max_steps if training_args.max_steps>0 else 'epochs=' + str(CFG.num_train_epochs)}")


## 12 — Train (resume-aware, Drive-backed)

This cell is safe to rerun after any disconnect — it automatically resumes from `latest_ckpt` if found.

- First run: `trainer.train()` from scratch, saving to `CHECKPOINT_DIR/checkpoint-25`, `checkpoint-50`, ...
- After disconnect: re-run notebook top-to-bottom; this cell finds `latest_ckpt` and continues.
- On `KeyboardInterrupt` / Colab timeout, the last `save_steps` checkpoint is already on Drive.

In [ ]:
# Train — resume if checkpoint exists
try:
    if latest_ckpt and Path(latest_ckpt).exists():
        print(f"Resuming from {latest_ckpt}")
        trainer.train(resume_from_checkpoint=latest_ckpt)
    else:
        print("Starting fresh training...")
        trainer.train()
except Exception as e:
    print(f"Training interrupted: {e}")
    # Save current state anyway
    try:
        trainer.save_state()
        print("Saved trainer state after interruption.")
    except Exception as se:
        print(f"Could not save state: {se}")
    raise

print("Training done.")
print(f"Final checkpoint dir: {CHECKPOINT_DIR}")
!ls -lh "{CHECKPOINT_DIR}" 2>&1 | head -n 50
if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")


## 13 — Save final LoRA adapter to Drive (and optionally Hub)

Only the adapter (~30–80 MB) is saved, not the full 2B base. This adapter path is what `backend/config.py` and stage 2/3 notebooks will load.

- The adapter is saved to `ADAPTER_OUTPUT_DIR` on Drive and also to `training/adapters/` locally if present.
- To push to Hugging Face Hub: set `PUSH_TO_HUB = True` and fill `HF_REPO_ID`.

In [ ]:
from pathlib import Path

PUSH_TO_HUB = False
HF_REPO_ID = ""  # e.g. "your-username/satquery-qwen2vl-stage1-bigearthnet"

print(f"Saving adapter to {ADAPTER_OUTPUT_DIR}")
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUTPUT_DIR))
processor.tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))
# Also save processor config
try:
    processor.save_pretrained(str(ADAPTER_OUTPUT_DIR))
except Exception as e:
    print(f"processor.save_pretrained skipped: {e}")
print(f"Saved adapter files:")
!ls -lh "{ADAPTER_OUTPUT_DIR}" 2>&1 | head -n 30

# Also cache a copy under repo training/adapters for local backend reference (gitignored)
try:
    local_adapter = Path("training/adapters/stage1_bigearthnet_qlora")
    # Don't copy weights to git — just note the Drive path
    local_adapter.mkdir(parents=True, exist_ok=True)
    (local_adapter / "DRIVE_PATH.txt").write_text(str(ADAPTER_OUTPUT_DIR))
    print(f"Wrote Drive pointer to {local_adapter / 'DRIVE_PATH.txt'}")
except Exception as e:
    print(f"Local pointer skipped: {e}")

if PUSH_TO_HUB and HF_REPO_ID:
    from huggingface_hub import HfApi
    api = HfApi()
    print(f"Pushing adapter to Hub: {HF_REPO_ID}")
    model.push_to_hub(HF_REPO_ID)
    processor.push_to_hub(HF_REPO_ID)
    print("Pushed to Hub — set this repo id in backend/config.py ADAPTER_PATH")
else:
    print("Hub push skipped (set PUSH_TO_HUB=True to enable).")
    print(f"\nFor backend inference, set in backend/config.py:")
    print(f"  ADAPTER_PATH = \"{ADAPTER_OUTPUT_DIR}\"  # or HF_REPO_ID if pushed")


## 14 — Sanity-check inference (adapter + base)

Loads the freshly saved adapter on top of the 4-bit base and runs a captioning query on a held-out image. No extra training — just verification that the adapter loads for `backend/models/` inference.

If this fails with OOM, reduce `max_new_tokens` or restart runtime (model is still on GPU from training).


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from pathlib import Path

# Free training model from GPU first (to avoid double-load OOM)
try:
    del trainer, model
    torch.cuda.empty_cache()
    print("Cleared training model from GPU.")
except: pass

adapter_path = str(ADAPTER_OUTPUT_DIR)
base_id = CFG.base_model
print(f"Loading base {base_id} for inference...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = Qwen2VLForConditionalGeneration.from_pretrained(base_id, quantization_config=bnb, device_map="auto", trust_remote_code=True)
proc = AutoProcessor.from_pretrained(base_id, min_pixels=CFG.image_min_pixels, max_pixels=CFG.image_max_pixels, trust_remote_code=True)
print(f"Loading adapter {adapter_path}...")
peft_model = PeftModel.from_pretrained(base, adapter_path)
peft_model.eval()
print("Adapter loaded.")

# Pick a held-out image
test_ex = datasetDict["train"][0] if len(datasetDict["train"])>0 else datasetDict["test"][0]
img = test_ex["image"]
question = "Describe the land cover in this satellite image."
messages = [{"role":"user","content":[{"type":"image","image": img},{"type":"text","text": question}]}]
text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = proc(text=[text], images=[img], return_tensors="pt", padding=True).to(peft_model.device)
print(f"Prompt: {question}")
display(img.resize((224,224)))

with torch.no_grad():
    out = peft_model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    # Trim prompt tokens
    gen = out[0][inputs["input_ids"].shape[1]:]
    answer = proc.decode(gen, skip_special_tokens=True)
    print(f"\nModel answer: {answer}")
    print(f"\nGround truth: {test_ex['answer']}")


## 15 — Next stages (copy this notebook)

For **Stage 2 (VRSBench/RSVQA SFT)** and **Stage 3 (CDVQA change SFT)**: duplicate this notebook, then change only:

1. `CHECKPOINT_DIR = DRIVE_ROOT/checkpoints/stage2_vrsbench_sft`
2. `CFG.adapter_to_continue = str(DRIVE_ROOT/checkpoints/stage1_bigearthnet_qlora/final_adapter)` — starts from stage 1 adapter instead of base
3. `DATASET_CACHE_DIR`, `subset_size`, and the dataset builder (swap `build_synthetic_dataset` for `vrsbench`/`cdvqa` loader — same `image`/`question`/`answer` schema)
4. Optionally lower `learning_rate` to `1e-4` for SFT stages

Backend inference (`backend/models/` + `backend/registry.py`) loads the final stage adapter via `PeftModel.from_pretrained(base, adapter_path)` — see `backend/config.py:ADAPTER_PATH`.

---
**Troubleshooting appendix**
- `OutOfMemoryError` → lower `max_seq_length` to 768, or `image_max_pixels` to `384*28*28`, or set `preprocess` truncation earlier.
- `Tokenizer` warnings about `pad_token` → already handled by `DataCollatorForSeq2Seq(pad_to_multiple_of=8)`; safe to ignore.
- `Drive quota exceeded` → delete old `checkpoint-*` folders, keep only last 2 (`save_total_limit=2` does this).
- `Model not found` / 401 → `huggingface-cli login` or set `HF_TOKEN` env if base is gated.
